# Feature Engineering – Practical Machine Learning Series

## Learning objectives
By the end of this notebook, you should be able to explain the concept, identify appropriate techniques, implement them with pandas/scikit-learn, and recognize common risks.

**Important:** Feature engineering must be learned from the training data only when a transformation learns statistics or target relationships. Fit transformations on training data and apply the fitted transformation to validation/test data.

# 03 – Categorical Feature Engineering

Categorical variables represent groups or labels. Most machine-learning estimators require numeric input, so categories often need encoding. Choose an encoding based on category meaning, cardinality, data size, model type, and leakage risk.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.DataFrame({
    "plan": ["Basic", "Premium", "Standard", "Basic", "Premium", "Standard", "Basic", "Premium"],
    "city": ["Salem", "Chennai", "Madurai", "Salem", "Coimbatore", "Chennai", "Madurai", "Salem"],
    "satisfaction": ["Low", "High", "Medium", "Medium", "High", "Low", "Low", "High"],
    "is_active": ["Yes", "Yes", "No", "Yes", "No", "No", "Yes", "Yes"],
    "churn": [1, 0, 0, 1, 0, 1, 1, 0]
})
display(df)

## 1. Types of categorical variables

- **Nominal:** no natural order (city, product type, color).
- **Ordinal:** meaningful order (low, medium, high).
- **Binary:** exactly two meaningful states (yes/no).
- **Cardinality:** number of distinct categories. High-cardinality columns may have hundreds or thousands of values.

Do not assign numeric order to nominal categories unless the numeric values are only identifiers and are handled appropriately.

## 2. Encoding methods: comparison

| Method | How it works | Suitable use |
|---|---|---|
| One-hot | One indicator column per category | Low/medium-cardinality nominal features |
| Ordinal | Maps ordered labels to ordered integers | Categories with genuine order |
| Label encoding | Maps labels to integer IDs | Often suitable for target labels; for input features use only when model/encoding makes sense |
| Frequency | Replaces category with its count or proportion | High-cardinality categories when frequency is informative |
| Target | Uses target statistics per category | High-cardinality features; requires out-of-fold/train-only safeguards |
| Rare grouping | Combines infrequent values into `Other` | Sparse categories and noisy long tails |

One-hot encoding does not imply an order. Ordinal encoding does. Frequency encoding captures prevalence, not necessarily business value.

In [ ]:
# One-hot encoding for nominal columns
one_hot = pd.get_dummies(df[["plan", "city"]], dtype=int)
display(one_hot.head())

# Ordinal encoding: order must be defined by domain knowledge
order = {"Low": 0, "Medium": 1, "High": 2}
df["satisfaction_ordinal"] = df["satisfaction"].map(order)
df["active_binary"] = df["is_active"].map({"No": 0, "Yes": 1})
display(df[["satisfaction", "satisfaction_ordinal", "is_active", "active_binary"]])

## 3. Label encoding vs ordinal encoding

**Label encoding** is a general label-to-integer mapping. The assigned numbers may be arbitrary and can accidentally suggest order to a model.

**Ordinal encoding** deliberately maps categories according to a real ranking, such as Bronze < Silver < Gold. Document the order and define how unknown values are handled.

For input columns, one-hot or another appropriate representation is usually safer for nominal values. Label encoding is commonly used for classification target labels.

In [ ]:
# Frequency encoding: use training data to learn category frequencies.
train_cities = df["city"].iloc[:6]
freq_map = train_cities.value_counts(normalize=True)
df["city_frequency"] = df["city"].map(freq_map).fillna(0)
display(df[["city", "city_frequency"]])

## 4. Target encoding and leakage protection

Target encoding replaces a category with a target-derived statistic, such as the mean target value for that category. It can be useful for high-cardinality features, but it can overfit—especially for rare categories.

Safe practice:
- split data before fitting target encoders;
- use out-of-fold encoding for training rows;
- smooth category estimates toward the global mean;
- use only training targets to encode validation/test rows;
- handle unseen categories with a global fallback.

Never calculate target means on the full dataset before splitting.

In [ ]:
# Conceptual illustration only: this is NOT a production-safe training encoder.
# It demonstrates the statistic being estimated.
global_rate = df["churn"].mean()
target_map = df.groupby("city")["churn"].mean()
print("Global churn rate:", round(global_rate, 3))
print("City target means (illustration):")
display(target_map)
print("For a real model, use cross-fitted target encoding inside the training workflow.")

## 5. Rare categories, unknown categories, and combining categories

- **Rare category grouping:** group values below a frequency threshold into `Other`. Choose the threshold using training data.
- **Unknown categories:** categories that appear during validation/production but were absent during fitting. Use a documented fallback.
- **Combining categories:** merge values only when domain meaning supports it (e.g. product subtypes into a broader product family), not just to improve a score.
- **High cardinality:** consider frequency encoding, hashing, carefully cross-fitted target encoding, or domain-based grouping.

For scikit-learn, `OneHotEncoder(handle_unknown="ignore")` prevents errors for unseen categories.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

X = df[["plan", "city", "satisfaction_ordinal", "active_binary"]]
cat_cols = ["plan", "city"]
num_cols = ["satisfaction_ordinal", "active_binary"]

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="most_frequent"))]), num_cols)
])

# Fit/transform demonstration. In a real project, split X/y first and fit on X_train only.
encoded = preprocessor.fit_transform(X)
print("Encoded matrix shape:", encoded.shape)